# Fine-tune SBERT trên Essays Dataset cho Personality Retrieval (Kaggle)

**Optimized cho Kaggle GPU**: T4 ×2 (16GB ×2) hoặc P100 (16GB).

**Mục tiêu**: Fine-tune sentence embedding model để embed essays sao cho các essays có cùng nhãn OCEAN nằm gần nhau trong embedding space — phục vụ retrieval cho ICL + RAG pipeline.

**Approach**: Multiple Negatives Ranking Loss với in-batch negatives.

**Base model**: `nomic-ai/nomic-embed-text-v1.5`
- Context dài (chỉnh xuống 1024 cho phù hợp 16GB GPU)
- Pre-trained contrastive trên 235M pairs
- Apache 2.0

**Anti-leakage**: Test set không load trong notebook này.

## 1. Cài đặt

In [ ]:
!pip install -q -U sentence-transformers==3.* faiss-cpu einops

## 2. CONFIG — Sửa các path và hyperparameters ở đây

Mọi thứ cần điều chỉnh đều nằm trong cell này.

In [ ]:
import os
from pathlib import Path

# ════════════════════════════════════════════════════════════════════════════
# DATA PATHS — sửa ở đây
# ════════════════════════════════════════════════════════════════════════════

# Trên Kaggle, dataset được mount tại /kaggle/input/<dataset-slug>/
# Sửa DATA_DIR thành đường dẫn tới folder chứa 3 file CSV của bạn.
#
# Ví dụ:
#   DATA_DIR = "/kaggle/input/essays-bigfive-splits"
#   DATA_DIR = "./data"   # khi chạy local
#
DATA_DIR = "/kaggle/input/essays-bigfive-splits"

TRAIN_FILENAME = "train.csv"
VAL_FILENAME   = "val.csv"
# TEST không dùng ở đây để tránh leakage

# ════════════════════════════════════════════════════════════════════════════
# OUTPUT PATHS
# ════════════════════════════════════════════════════════════════════════════
# Trên Kaggle dùng /kaggle/working/ để được persist khi save notebook
WORK_DIR        = "/kaggle/working" if Path("/kaggle/working").exists() else "."
MODEL_OUTPUT_DIR = f"{WORK_DIR}/sbert_essays_finetuned"
ARTIFACTS_DIR    = f"{WORK_DIR}/rag_artifacts"

# ════════════════════════════════════════════════════════════════════════════
# MODEL CONFIG
# ════════════════════════════════════════════════════════════════════════════
BASE_MODEL  = "nomic-ai/nomic-embed-text-v1.5"
MAX_SEQ_LEN = 1024   # 1024 đủ cho ~95% essays. Giảm xuống 512 nếu OOM, tăng lên 2048 nếu GPU rộng

# ════════════════════════════════════════════════════════════════════════════
# TRAINING CONFIG — đã tune cho 16GB GPU
# ════════════════════════════════════════════════════════════════════════════
BATCH_SIZE   = 8     # với MAX_SEQ_LEN=1024, fp16 → fit ~14GB. Giảm xuống 4 nếu OOM
NUM_EPOCHS   = 4
LR           = 2e-5
WARMUP_RATIO = 0.1
USE_FP16     = True  # bắt buộc trên T4/P100 để fit

# Pair sampling
PAIRS_PER_ANCHOR = 3
MAX_PAIRS        = 6000

# ════════════════════════════════════════════════════════════════════════════
# SEED
# ════════════════════════════════════════════════════════════════════════════
SEED = 42

# ════════════════════════════════════════════════════════════════════════════
# Setup
# ════════════════════════════════════════════════════════════════════════════
TRAIN_CSV = os.path.join(DATA_DIR, TRAIN_FILENAME)
VAL_CSV   = os.path.join(DATA_DIR, VAL_FILENAME)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

# Force single GPU (T4×2 cần DDP, không setup ở notebook này)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

TRAITS = ["cOPN", "cCON", "cEXT", "cAGR", "cNEU"]
TRAIT_NAMES = {
    "cOPN": "Openness",
    "cCON": "Conscientiousness",
    "cEXT": "Extraversion",
    "cAGR": "Agreeableness",
    "cNEU": "Neuroticism",
}

# Verify paths
print("=" * 60)
print("PATH CHECK")
print("=" * 60)
for name, p in [("TRAIN_CSV", TRAIN_CSV), ("VAL_CSV", VAL_CSV)]:
    exists = "✓" if Path(p).exists() else "✗ MISSING"
    print(f"  {name:12s} {exists}  {p}")
print(f"\n  MODEL_OUTPUT_DIR: {MODEL_OUTPUT_DIR}")
print(f"  ARTIFACTS_DIR:    {ARTIFACTS_DIR}")

## 3. Imports & GPU Check

In [ ]:
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
)
from sentence_transformers.evaluation import TripletEvaluator

import faiss
from sklearn.metrics import accuracy_score, f1_score

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU info
print("=" * 60)
print("GPU CHECK")
print("=" * 60)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}, {props.total_memory/1e9:.1f}GB")
    device = "cuda"
else:
    print("  No GPU detected — fine-tuning sẽ rất chậm trên CPU")
    device = "cpu"
print(f"\n  Using device: {device}")
print(f"  FP16 enabled: {USE_FP16}")

## 4. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)

for trait in TRAITS:
    train_df[trait] = train_df[trait].astype(int)
    val_df[trait]   = val_df[trait].astype(int)

# Text length stats — quan trọng để verify MAX_SEQ_LEN
train_df["_n_words"] = train_df["text"].str.split().str.len()
n_words = train_df["_n_words"]

print(f"Train size: {len(train_df)} | Val size: {len(val_df)}")
print(f"\nWord count stats (train):")
print(f"  mean: {n_words.mean():.0f}, median: {n_words.median():.0f}")
print(f"  p90:  {n_words.quantile(0.9):.0f}, p95: {n_words.quantile(0.95):.0f}, max: {n_words.max():.0f}")
print(f"\n  Approx tokens (× 1.3): p95 ≈ {n_words.quantile(0.95) * 1.3:.0f}")
print(f"  -> MAX_SEQ_LEN={MAX_SEQ_LEN} {'OK ✓' if MAX_SEQ_LEN >= n_words.quantile(0.95) * 1.3 else '⚠ có thể truncate nhiều'}")

print(f"\nLabel distribution (train):")
for trait in TRAITS:
    vc = train_df[trait].value_counts()
    print(f"  {TRAIT_NAMES[trait]:20s}: High={vc.get(1,0):4d}, Low={vc.get(0,0):4d}")

train_df.drop(columns=["_n_words"], inplace=True)

## 5. Build Training Pairs (cho MultipleNegativesRankingLoss)

Với mỗi anchor và mỗi trait, lấy 1 essay khác **cùng nhãn trait đó** làm positive. Trong batch, MNRL tự động dùng các sample khác làm negatives.

In [ ]:
def build_positive_pairs(
    df: pd.DataFrame,
    traits: list,
    pairs_per_anchor: int = PAIRS_PER_ANCHOR,
    max_pairs: int = MAX_PAIRS,
    seed: int = SEED,
) -> list:
    rng = random.Random(seed)
    texts = df["text"].tolist()
    n = len(df)
    
    idx_by_trait_label = {}
    for trait in traits:
        labels = df[trait].tolist()
        idx_by_trait_label[trait] = {
            0: [i for i, v in enumerate(labels) if v == 0],
            1: [i for i, v in enumerate(labels) if v == 1],
        }
    
    pairs = []
    indices = list(range(n))
    rng.shuffle(indices)
    
    for anchor_idx in indices:
        anchor_text = texts[anchor_idx]
        for trait in traits:
            anchor_label = df[trait].iloc[anchor_idx]
            pos_pool = [i for i in idx_by_trait_label[trait][anchor_label] if i != anchor_idx]
            if not pos_pool:
                continue
            sampled = rng.sample(pos_pool, min(pairs_per_anchor, len(pos_pool)))
            for pos_idx in sampled:
                pairs.append(InputExample(texts=[
                    f"search_document: {anchor_text}",
                    f"search_document: {texts[pos_idx]}",
                ]))
                if len(pairs) >= max_pairs:
                    rng.shuffle(pairs)
                    return pairs
    
    rng.shuffle(pairs)
    return pairs


train_pairs = build_positive_pairs(train_df, TRAITS)
print(f"Training pairs: {len(train_pairs)}")

## 6. Build Validation Triplets (cho TripletEvaluator → early stopping)

In [ ]:
def build_eval_triplets(df, traits, n_per_trait=80, seed=SEED):
    rng = random.Random(seed)
    texts = df["text"].tolist()
    anchors, positives, negatives = [], [], []
    
    for trait in traits:
        labels = df[trait].tolist()
        idx_pos = [i for i, v in enumerate(labels) if v == 1]
        idx_neg = [i for i, v in enumerate(labels) if v == 0]
        if len(idx_pos) < 2 or len(idx_neg) < 2:
            continue
        
        for k in range(n_per_trait):
            if k % 2 == 0:
                anchor_idx = rng.choice(idx_pos)
                pos_pool = [i for i in idx_pos if i != anchor_idx]
                neg_pool = idx_neg
            else:
                anchor_idx = rng.choice(idx_neg)
                pos_pool = [i for i in idx_neg if i != anchor_idx]
                neg_pool = idx_pos
            
            if not pos_pool or not neg_pool:
                continue
            
            anchors.append(f"search_document: {texts[anchor_idx]}")
            positives.append(f"search_document: {texts[rng.choice(pos_pool)]}")
            negatives.append(f"search_document: {texts[rng.choice(neg_pool)]}")
    
    return anchors, positives, negatives


val_anchors, val_pos, val_neg = build_eval_triplets(val_df, TRAITS, n_per_trait=80)
print(f"Validation triplets: {len(val_anchors)}")

evaluator = TripletEvaluator(
    anchors=val_anchors,
    positives=val_pos,
    negatives=val_neg,
    name="essays-val",
    show_progress_bar=False,
    batch_size=BATCH_SIZE,
)

## 7. Load Base Model

In [ ]:
model = SentenceTransformer(
    BASE_MODEL,
    trust_remote_code=True,
    device=device,
)
model.max_seq_length = MAX_SEQ_LEN

print(f"Model loaded: {BASE_MODEL}")
print(f"Max seq length: {model.max_seq_length}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

print("\n→ Baseline evaluation (trước fine-tune)...")
baseline_score = evaluator(model)
print(f"Baseline triplet accuracy: {baseline_score}")

## 8. Fine-tune

Note: trên T4/P100 với MAX_SEQ_LEN=1024 và batch=8, một epoch ≈ 4-8 phút. Tổng 4 epoch ≈ 20-30 phút.

In [ ]:
train_loader = DataLoader(train_pairs, batch_size=BATCH_SIZE, shuffle=True)
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(len(train_loader) * NUM_EPOCHS * WARMUP_RATIO)
eval_steps = max(50, len(train_loader) // 2)

print(f"Steps per epoch: {len(train_loader)}")
print(f"Total steps:     {len(train_loader) * NUM_EPOCHS}")
print(f"Warmup steps:    {warmup_steps}")
print(f"Eval every:      {eval_steps} steps\n")

model.fit(
    train_objectives=[(train_loader, train_loss)],
    evaluator=evaluator,
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={"lr": LR},
    evaluation_steps=eval_steps,
    save_best_model=True,
    output_path=MODEL_OUTPUT_DIR,
    show_progress_bar=True,
    use_amp=USE_FP16,
)

# Free memory
del model, train_loss, train_loader
torch.cuda.empty_cache()

## 9. Load Best Checkpoint & Evaluate

In [ ]:
best_model = SentenceTransformer(
    MODEL_OUTPUT_DIR,
    trust_remote_code=True,
    device=device,
)
best_model.max_seq_length = MAX_SEQ_LEN

final_score = evaluator(best_model)
print(f"=== Triplet accuracy ===")
print(f"Before fine-tune: {baseline_score:.4f}")
print(f"After  fine-tune: {final_score:.4f}")
print(f"Δ:                {final_score - baseline_score:+.4f}")

## 10. Encode + Build FAISS Index

In [ ]:
ENCODE_BATCH = 16  # encode batch lớn hơn fit batch vì không cần backward

def encode_essays(model, texts, batch_size=ENCODE_BATCH, prefix="search_document: "):
    prefixed = [prefix + t for t in texts]
    return model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

print("Encoding train set...")
train_emb = encode_essays(best_model, train_df["text"].tolist())
print(f"  shape: {train_emb.shape}")

print("\nEncoding val set...")
val_emb = encode_essays(best_model, val_df["text"].tolist())
print(f"  shape: {val_emb.shape}")

# FAISS index (cosine = inner product trên normalized vectors)
dim = train_emb.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(train_emb.astype(np.float32))
print(f"\nFAISS index: {index.ntotal} vectors, dim={dim}")

## 11. Sanity Check: Retrieval-based Label Accuracy

Proxy metric — không phải downstream LLM accuracy.

In [ ]:
TOP_K = 5
_, neighbor_idx = index.search(val_emb.astype(np.float32), TOP_K)

print(f"=== Retrieval-based label accuracy (top-{TOP_K} majority vote) ===\n")
for trait in TRAITS:
    train_labels = train_df[trait].values
    val_labels   = val_df[trait].values
    
    preds = [int(train_labels[neighbor_idx[i]].mean() >= 0.5) for i in range(len(val_df))]
    
    acc = accuracy_score(val_labels, preds)
    f1  = f1_score(val_labels, preds, average="macro")
    print(f"  {TRAIT_NAMES[trait]:20s} acc={acc:.4f}  macro-F1={f1:.4f}")

## 12. Ablation: Base Model vs Fine-tuned

In [ ]:
# Free fine-tuned model trước khi load base
del best_model
torch.cuda.empty_cache()

print("Loading base model for comparison...")
base_model = SentenceTransformer(BASE_MODEL, trust_remote_code=True, device=device)
base_model.max_seq_length = MAX_SEQ_LEN

print("Encoding với base model...")
train_emb_base = encode_essays(base_model, train_df["text"].tolist())
val_emb_base   = encode_essays(base_model, val_df["text"].tolist())

index_base = faiss.IndexFlatIP(dim)
index_base.add(train_emb_base.astype(np.float32))
_, neighbor_idx_base = index_base.search(val_emb_base.astype(np.float32), TOP_K)

print(f"\n=== So sánh retrieval-based accuracy (top-{TOP_K}) ===\n")
print(f"{'Trait':22s} {'Base':>10s} {'Fine-tuned':>12s} {'Δ':>10s}")
print("-" * 56)
for trait in TRAITS:
    train_labels = train_df[trait].values
    val_labels   = val_df[trait].values
    
    preds_base = [int(train_labels[neighbor_idx_base[i]].mean() >= 0.5) for i in range(len(val_df))]
    preds_ft   = [int(train_labels[neighbor_idx[i]].mean() >= 0.5)      for i in range(len(val_df))]
    
    acc_base = accuracy_score(val_labels, preds_base)
    acc_ft   = accuracy_score(val_labels, preds_ft)
    
    print(f"{TRAIT_NAMES[trait]:22s} {acc_base:>10.4f} {acc_ft:>12.4f} {acc_ft-acc_base:>+10.4f}")

del base_model, train_emb_base, val_emb_base, index_base
torch.cuda.empty_cache()

## 13. Save Artifacts cho RAG Pipeline

In [ ]:
# FAISS index
faiss.write_index(index, os.path.join(ARTIFACTS_DIR, "train_index.faiss"))

# Embeddings + metadata
np.save(os.path.join(ARTIFACTS_DIR, "train_embeddings.npy"), train_emb)
train_df.to_csv(os.path.join(ARTIFACTS_DIR, "train_metadata.csv"), index=False)

# Config snapshot
import json
config_snapshot = {
    "base_model": BASE_MODEL,
    "max_seq_len": MAX_SEQ_LEN,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "lr": LR,
    "pairs_per_anchor": PAIRS_PER_ANCHOR,
    "n_train_pairs": len(train_pairs),
    "baseline_triplet_acc": float(baseline_score),
    "final_triplet_acc": float(final_score),
    "embedding_dim": dim,
    "prefix": "search_document: ",
}
with open(os.path.join(ARTIFACTS_DIR, "config.json"), "w") as f:
    json.dump(config_snapshot, f, indent=2)

print(f"=== Artifacts saved to {ARTIFACTS_DIR}/ ===")
for fname in sorted(os.listdir(ARTIFACTS_DIR)):
    fpath = os.path.join(ARTIFACTS_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {fname:25s} {size_mb:>8.2f} MB")

print(f"\n=== Fine-tuned model saved to {MODEL_OUTPUT_DIR}/ ===")
print(f"  Total size: {sum(os.path.getsize(os.path.join(MODEL_OUTPUT_DIR, f)) for f in os.listdir(MODEL_OUTPUT_DIR) if os.path.isfile(os.path.join(MODEL_OUTPUT_DIR, f))) / 1e6:.0f} MB")

## 14. Code mẫu để dùng trong Pipeline Detection

```python
from sentence_transformers import SentenceTransformer
import faiss, pandas as pd, numpy as np

# Load artifacts
model = SentenceTransformer(
    "/kaggle/working/sbert_essays_finetuned",
    trust_remote_code=True,
    device="cuda",
)
model.max_seq_length = 1024

index = faiss.read_index("/kaggle/working/rag_artifacts/train_index.faiss")
train_meta = pd.read_csv("/kaggle/working/rag_artifacts/train_metadata.csv")

# Encode test essays
test_df = pd.read_csv("test.csv")
test_texts = ["search_document: " + t for t in test_df["text"]]
test_emb = model.encode(test_texts, normalize_embeddings=True, convert_to_numpy=True)

# Retrieve top-k
K = 5
scores, idx = index.search(test_emb.astype(np.float32), K)

# idx[i] = indices of top-K train essays for test essay i
# Dùng train_meta.iloc[idx[i]] để lấy text + labels làm ICL examples
```

---

## Bước tiếp theo

1. ✅ **Bước này**: Fine-tune trên Essays (đã xong)
2. ⏭ **Bước 2**: Sequential — pre-finetune trên PANDORA → fine-tune Essays
3. ⏭ **Bước 3**: Per-trait fine-tuning (5 model riêng)
4. ⏭ **Bước 4**: Plug FAISS index vào DTI pipeline, đo F1 High/Low downstream